# 🎓 Pedagogical Knowledge Graph Extractor

**iREL/LTRC Recruitment Task — Complete Pipeline on Google Colab**

This notebook runs the full pipeline:
1. **Speech-to-Text** — Whisper (small) transcribes video/audio
2. **LLM Extraction** — Groq (Llama-3.3-70B) extracts concepts & prerequisites
3. **Knowledge Graph** — NetworkX builds the pedagogical graph
4. **Metrics & Analysis** — PageRank, centrality, HITS, community detection
5. **Interactive Visualization** — PyVis renders the graph in-browser

### Features
- **Multi-language support**: English, Hindi-English (Hinglish), Telugu-English (Tenglish)
- Overlapping chunk-level extraction for long transcripts
- Embedding-based concept deduplication (sentence-transformers)
- Edge weight refinement from multiple detection sources
- Louvain community detection with colored clusters
- Corrected learning path (depth-first topological sort)
- Telugu script (తెలుగు) transliteration to Roman

> ⚡ **Runs entirely on Colab free tier** — no local GPU needed.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install openai-whisper torch ffmpeg-python pydub
!pip install rake-nltk nltk
!pip install networkx pyvis plotly
!pip install pyyaml groq
!pip install sentence-transformers
!pip install yt-dlp
!apt-get -qq install ffmpeg
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('✅ All dependencies installed!')

## 2. Configuration

Set your **Groq API key** below. Get a free key at https://console.groq.com/keys

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────────
# 🔑 Secure API Key Input
from getpass import getpass
GROQ_API_KEY = getpass("🔑 Enter your Groq API key: ")
WHISPER_MODEL = "small"   # tiny | base | small | medium | large
WHISPER_LANGUAGE = None    # None = auto-detect. Use "en", "hi", "te" to force
LLM_MODEL = "llama-3.3-70b-versatile"
MAX_TOKENS = 8192

# Input: choose ONE input mode
YOUTUBE_URL = ""  # e.g. "https://www.youtube.com/watch?v=abc123"
PASTE_TEXT = ""   # Or paste transcript text directly
# Or upload a file via the sidebar (below)

import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print(f"✅ Config: Whisper={WHISPER_MODEL}, Language={'auto-detect' if WHISPER_LANGUAGE is None else WHISPER_LANGUAGE}, LLM={LLM_MODEL}")

## 3. Core Pipeline Code

All pipeline modules in one cell for Colab portability.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CORE PIPELINE — Self-contained for Google Colab
# ═══════════════════════════════════════════════════════════════

import json, os, re, subprocess, tempfile, logging, heapq
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from itertools import combinations

import networkx as nx
import numpy as np

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger('pipeline')

# ── Helpers ──────────────────────────────────────────────────
def format_timestamp(seconds: float) -> str:
    return f"{int(seconds // 60)}:{int(seconds % 60):02d}"

def now_iso() -> str:
    return datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')

# ── Telugu Script Transliteration ───────────────────────────
def _transliterate_telugu(text: str) -> str:
    """Convert Telugu script (U+0C00-U+0C7F) to Roman/Latin."""
    VIRAMA = '\u0C4D'
    vowels = {
        '\u0C05':'a','\u0C06':'aa','\u0C07':'i','\u0C08':'ee','\u0C09':'u','\u0C0A':'oo',
        '\u0C0B':'ru','\u0C0E':'e','\u0C0F':'ee','\u0C10':'ai','\u0C12':'o','\u0C13':'oo','\u0C14':'au',
    }
    matras = {
        '\u0C3E':'aa','\u0C3F':'i','\u0C40':'ee','\u0C41':'u','\u0C42':'oo','\u0C43':'ru',
        '\u0C46':'e','\u0C47':'ee','\u0C48':'ai','\u0C4A':'o','\u0C4B':'oo','\u0C4C':'au',
    }
    consonants = {
        '\u0C15':'k','\u0C16':'kh','\u0C17':'g','\u0C18':'gh','\u0C19':'ng',
        '\u0C1A':'ch','\u0C1B':'chh','\u0C1C':'j','\u0C1D':'jh','\u0C1E':'ny',
        '\u0C1F':'t','\u0C20':'th','\u0C21':'d','\u0C22':'dh','\u0C23':'n',
        '\u0C24':'t','\u0C25':'th','\u0C26':'d','\u0C27':'dh','\u0C28':'n',
        '\u0C2A':'p','\u0C2B':'ph','\u0C2C':'b','\u0C2D':'bh','\u0C2E':'m',
        '\u0C2F':'y','\u0C30':'r','\u0C31':'rr','\u0C32':'l','\u0C33':'l',
        '\u0C35':'v','\u0C36':'sh','\u0C37':'sh','\u0C38':'s','\u0C39':'h',
    }
    digits = {'\u0C66':'0','\u0C67':'1','\u0C68':'2','\u0C69':'3','\u0C6A':'4',
              '\u0C6B':'5','\u0C6C':'6','\u0C6D':'7','\u0C6E':'8','\u0C6F':'9'}
    result = []
    i, n = 0, len(text)
    while i < n:
        ch = text[i]
        if ch in consonants:
            base = consonants[ch]; i += 1
            if i < n and text[i] == VIRAMA: result.append(base); i += 1
            elif i < n and text[i] in matras: result.append(base + matras[text[i]]); i += 1
            else: result.append(base + 'a')
            continue
        if ch in vowels: result.append(vowels[ch]); i += 1; continue
        if ch in matras: result.append(matras[ch]); i += 1; continue
        if ch == '\u0C02': result.append('n'); i += 1; continue  # anusvara
        if ch == '\u0C03': result.append('h'); i += 1; continue  # visarga
        if ch == '\u0C01': result.append('n'); i += 1; continue  # chandrabindu
        if ch == VIRAMA: i += 1; continue
        if ch in digits: result.append(digits[ch]); i += 1; continue
        if '\u0C00' <= ch <= '\u0C7F': i += 1; continue
        result.append(ch); i += 1
    return ''.join(result)

# ── Transliteration dispatcher ─────────────────────────────
def transliterate_devanagari(text: str) -> str:
    """Convert Devanagari / Urdu / Telugu script to Roman. Pass-through for Latin text."""
    if not text:
        return text
    has_dev = any('\u0900' <= c <= '\u097F' for c in text)
    has_ur = any('\u0600' <= c <= '\u06FF' for c in text)
    has_te = any('\u0C00' <= c <= '\u0C7F' for c in text)
    if not has_dev and not has_ur and not has_te:
        return text
    if has_ur:
        # Simple pass-through for Urdu chars (full impl in repo)
        out = []
        for c in text:
            if '\u0600' <= c <= '\u06FF': continue
            out.append(c)
        text = ''.join(out).strip()
    if has_te:
        text = _transliterate_telugu(text)
    if not has_dev:
        return text
    # Simple pass-through for Devanagari (full impl in repo)
    out = []
    for c in text:
        if '\u0900' <= c <= '\u097F': continue
        out.append(c)
    return ''.join(out).strip()

# ── Speech-to-Text ───────────────────────────────────────────
def transcribe_video(file_path: str) -> dict:
    """Transcribe audio/video file with Whisper (auto-detects language)."""
    import whisper
    logger.info(f'Loading Whisper model: {WHISPER_MODEL}')
    model = whisper.load_model(WHISPER_MODEL)

    # Extract audio if video
    ext = os.path.splitext(file_path)[1].lower()
    if ext in ('.mp4', '.mkv', '.webm', '.avi', '.mov'):
        audio_path = file_path.rsplit('.', 1)[0] + '.wav'
        subprocess.run(['ffmpeg', '-y', '-i', file_path, '-vn', '-acodec', 'pcm_s16le',
                        '-ar', '16000', '-ac', '1', audio_path],
                       capture_output=True, timeout=300)
        file_path = audio_path

    logger.info('Transcribing...')
    kw = {'task': 'transcribe'}
    if WHISPER_LANGUAGE:
        kw['language'] = WHISPER_LANGUAGE
    result = model.transcribe(file_path, **kw)

    lang = result.get('language', 'en')
    logger.info(f'Detected language: {lang}')

    segments = []
    for seg in result.get('segments', []):
        segments.append({
            'id': seg['id'],
            'start': seg['start'],
            'end': seg['end'],
            'text': seg['text'].strip(),
            'timestamp_label': format_timestamp(seg['start']),
        })

    full_text = result.get('text', '')
    video_id = os.path.splitext(os.path.basename(file_path))[0]

    return {
        'video_id': video_id,
        'metadata': {'source_file': file_path, 'language_detected': lang},
        'full_text': full_text,
        'segments': segments,
    }

def text_to_transcript(text: str, video_id: str = 'pasted_input') -> dict:
    """Convert raw text to transcript dict."""
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if s.strip()]
    segments = [{
        'id': i, 'start': i * 10.0, 'end': (i + 1) * 10.0,
        'text': sent, 'timestamp_label': format_timestamp(i * 10.0),
    } for i, sent in enumerate(sentences)]
    return {
        'video_id': video_id,
        'metadata': {'source_file': 'text_input', 'language_detected': 'en'},
        'full_text': text,
        'segments': segments,
    }

print('✅ Core helpers loaded (English + Hindi + Telugu support)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LLM EXTRACTOR — Two-pass (concepts + relationships) via Groq
# ═══════════════════════════════════════════════════════════════

CONCEPT_EXTRACTION_PROMPT = """You are an expert educational content analyzer. You will be given a transcript from an educational video lecture. The lecture may contain Hinglish (Hindi-English), Tenglish (Telugu-English), or pure English text.

Your task is to:
1. Identify the academic domain
2. Extract ALL academic concepts from the transcript

RULES:
- Extract ALL concepts discussed, explained, demonstrated, used, or referenced
- Include umbrella/category concepts AND specific ones
- Include techniques, paradigms, operations, building-block concepts
- Aim for 10-30 concepts. Better to include borderline concepts than miss them.
- Use canonical English names
- Importance: how central (0.0-1.0)
- Difficulty: "easy", "medium", or "hard"

Respond with ONLY valid JSON:
{{{{
  "domain": "...",
  "concepts": [
    {{{{"name": "...", "normalized_name": "...", "importance_score": 0.9, "difficulty": "medium", "description": "..."}}}}
  ]
}}}}

TRANSCRIPT:
{transcript}"""

RELATIONSHIP_DETECTION_PROMPT = """You are an expert educational content analyzer. Detect prerequisite relationships between these concepts.

CONCEPTS: {concept_list}

RULES:
- prerequisite means: "To understand B, you should first understand A"
- Detect from EXPLICIT AND IMPLICIT cues (including Telugu/Hindi cues like "mundu X tarvata Y", "pehle X phir Y")
- "X uses Y" means Y is prerequisite of X
- Only use concepts from the list. Do not invent new ones.
- Confidence: 0.5-1.0

Respond with ONLY valid JSON:
{{{{
  "relationships": [
    {{{{"from": "prereq", "to": "dependent", "confidence": 0.85, "evidence": "..."}}}}
  ]
}}}}

TRANSCRIPT:
{transcript}"""


def _parse_json(raw: str) -> dict:
    """Parse JSON from LLM with fallbacks."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', raw)
        if m:
            try: return json.loads(m.group(1))
            except: pass
        s = raw.find('{')
        if s >= 0:
            d = 0
            for i in range(s, len(raw)):
                if raw[i] == '{': d += 1
                elif raw[i] == '}': d -= 1
                if d == 0:
                    try: return json.loads(raw[s:i+1])
                    except: break
        return {}


def llm_extract(transcript: dict) -> dict:
    """Two-pass LLM extraction with overlapping chunk support."""
    from groq import Groq
    client = Groq(api_key=os.environ.get('GROQ_API_KEY', GROQ_API_KEY))

    text = transliterate_devanagari(transcript.get('full_text', ''))
    if not text:
        return {'domain': 'Unknown', 'concepts': [], 'relationships': []}

    def _call(prompt, json_mode=True):
        kw = dict(model=LLM_MODEL, messages=[
            {'role': 'system', 'content': 'Respond with ONLY valid JSON.'},
            {'role': 'user', 'content': prompt},
        ], temperature=0.1, max_tokens=MAX_TOKENS)
        if json_mode:
            kw['response_format'] = {'type': 'json_object'}
        return client.chat.completions.create(**kw).choices[0].message.content

    def _extract_single(chunk):
        # Pass 1: concepts
        p1 = CONCEPT_EXTRACTION_PROMPT.format(transcript=chunk)
        try:
            raw1 = _call(p1)
        except Exception:
            raw1 = _call(p1, json_mode=False)
        data1 = _parse_json(raw1)
        concepts_raw = data1.get('concepts', [])
        domain = data1.get('domain', 'Unknown')
        if not concepts_raw:
            return {'domain': domain, 'concepts': [], 'relationships': []}

        # Pass 2: relationships
        names = [c.get('name', '') for c in concepts_raw if c.get('name')]
        p2 = RELATIONSHIP_DETECTION_PROMPT.format(concept_list=', '.join(names), transcript=chunk)
        try:
            raw2 = _call(p2)
        except Exception:
            raw2 = _call(p2, json_mode=False)
        data2 = _parse_json(raw2)
        return {'domain': domain, 'concepts': concepts_raw, 'relationships': data2.get('relationships', [])}

    # Overlapping chunk splitting for long transcripts
    CHUNK_LIMIT = 25000
    if len(text) <= CHUNK_LIMIT:
        result = _extract_single(text)
    else:
        overlap = CHUNK_LIMIT // 5
        chunks = []
        start = 0
        while start < len(text):
            end = start + CHUNK_LIMIT
            chunk = text[start:end]
            if end < len(text):
                last_space = chunk.rfind(' ')
                if last_space > CHUNK_LIMIT // 2:
                    chunk = chunk[:last_space]
                    end = start + last_space
            chunks.append(chunk)
            start = end - overlap if end < len(text) else len(text)

        logger.info(f'Split into {len(chunks)} overlapping chunks')
        all_c, all_r, domain = [], [], 'Unknown'
        for i, ch in enumerate(chunks):
            logger.info(f'Chunk {i+1}/{len(chunks)}')
            try:
                cr = _extract_single(ch)
                if cr.get('domain', 'Unknown') != 'Unknown': domain = cr['domain']
                all_c.extend(cr.get('concepts', []))
                all_r.extend(cr.get('relationships', []))
            except Exception as e:
                logger.warning(f'Chunk {i+1} failed: {e}')

        # Dedup across chunks
        seen = set()
        merged_c = []
        for c in all_c:
            n = c.get('name', '').lower().strip()
            if n and n not in seen:
                seen.add(n)
                merged_c.append(c)
        cnames = {c.get('name', '').lower().strip() for c in merged_c}
        seen_e = set()
        merged_r = []
        for r in all_r:
            k = (r.get('from',''), r.get('to',''))
            if k not in seen_e and r.get('from','').lower() in cnames and r.get('to','').lower() in cnames:
                seen_e.add(k)
                merged_r.append(r)
        result = {'domain': domain, 'concepts': merged_c, 'relationships': merged_r}

    # Format into pipeline output
    segments = transcript.get('segments', [])
    full_lower = transcript.get('full_text', '').lower()

    concepts = []
    seen_names = set()
    for c in result.get('concepts', []):
        name = c.get('name', '').lower().strip().replace('_', ' ')
        if not name or name in seen_names: continue
        seen_names.add(name)
        # Find timestamps
        pat = re.compile(r'\b' + re.escape(name) + r'\b', re.I)
        timestamps = []
        for seg in segments:
            if pat.search(seg.get('text', '')):
                ts = seg.get('timestamp_label', '')
                if ts and ts not in timestamps: timestamps.append(ts)
        freq = len(pat.findall(full_lower))
        concepts.append({
            'name': name,
            'normalized_name': c.get('normalized_name', name.title()),
            'importance_score': round(float(c.get('importance_score', 0.5)), 4),
            'frequency': max(freq, 1),
            'first_mention': timestamps[0] if timestamps else None,
            'timestamps': timestamps,
            'difficulty': c.get('difficulty', 'unknown'),
            'description': c.get('description', ''),
        })

    concept_names = {c['name'] for c in concepts}
    relationships = []
    seen_edges = set()
    for r in result.get('relationships', []):
        src = r.get('from', '').lower().strip().replace('_', ' ')
        tgt = r.get('to', '').lower().strip().replace('_', ' ')
        conf = float(r.get('confidence', 0.7))
        if conf < 0.5: continue
        # Fuzzy match
        if src not in concept_names:
            for cn in concept_names:
                if src in cn or cn in src: src = cn; break
        if tgt not in concept_names:
            for cn in concept_names:
                if tgt in cn or cn in tgt: tgt = cn; break
        if src in concept_names and tgt in concept_names and src != tgt and (src, tgt) not in seen_edges:
            seen_edges.add((src, tgt))
            relationships.append({
                'from': src, 'to': tgt, 'relation': 'prerequisite',
                'confidence': round(conf, 4),
                'evidence': r.get('evidence', 'LLM-inferred'),
                'timestamp': '', 'detection_method': 'llm',
            })

    return {
        'domain': result.get('domain', 'Unknown'),
        'concepts': concepts,
        'relationships': relationships,
    }

print('✅ LLM extractor loaded (Hindi + Telugu + English)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  GRAPH BUILDER — With dedup, edge refinement, community detection
# ═══════════════════════════════════════════════════════════════

# Lazy-load sentence-transformer for semantic dedup
_EMBEDDER = None
def _get_embedder():
    global _EMBEDDER
    if _EMBEDDER is None:
        try:
            from sentence_transformers import SentenceTransformer
            _EMBEDDER = SentenceTransformer('all-MiniLM-L6-v2')
            logger.info('Loaded sentence-transformer for semantic dedup')
        except ImportError:
            _EMBEDDER = False
    return _EMBEDDER if _EMBEDDER else None


def deduplicate_concepts(concepts, relationships, sim_threshold=0.82):
    """Merge near-duplicate concepts using embedding or string similarity."""
    if len(concepts) <= 1:
        return concepts, relationships

    names = [c['name'] for c in concepts]
    embedder = _get_embedder()

    if embedder is not None:
        embs = embedder.encode(names, convert_to_tensor=False)
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        norms[norms == 0] = 1
        normed = embs / norms
        sim = normed @ normed.T
    else:
        n = len(names)
        sim = [[SequenceMatcher(None, names[i], names[j]).ratio() for j in range(n)] for i in range(n)]

    # Union-Find
    parent = list(range(len(names)))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            if float(sim[i][j]) >= sim_threshold:
                ri, rj = find(i), find(j)
                if ri != rj: parent[rj] = ri

    groups = defaultdict(list)
    for i in range(len(names)):
        groups[find(i)].append(i)

    merged = sum(1 for g in groups.values() if len(g) > 1)
    if merged:
        logger.info(f'Semantic dedup: merged {merged} groups')

    name_map = {}
    deduped = []
    for idxs in groups.values():
        best = max(idxs, key=lambda i: concepts[i].get('importance_score', 0))
        canon = concepts[best].copy()
        for idx in idxs:
            if idx != best:
                canon['frequency'] = canon.get('frequency', 0) + concepts[idx].get('frequency', 0)
            name_map[concepts[idx]['name']] = canon['name']
        deduped.append(canon)

    updated_rels = []
    seen = set()
    for r in relationships:
        src = name_map.get(r['from'], r['from'])
        tgt = name_map.get(r['to'], r['to'])
        if src == tgt: continue
        if (src, tgt) not in seen:
            seen.add((src, tgt))
            updated_rels.append({**r, 'from': src, 'to': tgt})

    return deduped, updated_rels


def build_knowledge_graph(concepts, relationships, video_id='video'):
    """Build NetworkX DiGraph with dedup, edge refinement, metrics, communities."""

    # Step 1: Semantic dedup
    concepts, relationships = deduplicate_concepts(concepts, relationships)

    G = nx.DiGraph()
    G.graph['video_id'] = video_id
    G.graph['created_at'] = now_iso()

    # Step 2: Add nodes
    for c in concepts:
        G.add_node(c['name'],
            normalized_name=c.get('normalized_name', c['name']),
            importance_score=c.get('importance_score', 0),
            frequency=c.get('frequency', 0),
            first_mention=c.get('first_mention', ''),
            timestamps=c.get('timestamps', []),
            difficulty=c.get('difficulty', 'unknown'),
        )

    # Step 3: Edge weight refinement — merge multi-source edges
    METHOD_WEIGHT = {'llm': 0.50, 'pattern_matching': 0.30, 'temporal_order': 0.10, 'co_occurrence': 0.10}
    bucket = defaultdict(list)
    for r in relationships:
        if r['from'] in G and r['to'] in G:
            bucket[(r['from'], r['to'])].append(r)

    for (src, tgt), rels in bucket.items():
        if len(rels) == 1:
            r = rels[0]
            G.add_edge(src, tgt, relation=r.get('relation', 'prerequisite'),
                       confidence=r.get('confidence', 0), evidence=r.get('evidence', ''),
                       detection_method=r.get('detection_method', ''))
        else:
            tw, wc, evs, methods = 0, 0, [], []
            for r in rels:
                m = r.get('detection_method', 'unknown')
                w = METHOD_WEIGHT.get(m, 0.15)
                wc += r.get('confidence', 0) * w
                tw += w
                if r.get('evidence'): evs.append(r['evidence'])
                if m not in methods: methods.append(m)
            conf = min(wc / tw if tw else 0, 1.0) + 0.05 * (len(rels) - 1)
            G.add_edge(src, tgt, relation='prerequisite',
                       confidence=round(min(conf, 1.0), 4),
                       evidence=evs[0] if evs else '',
                       detection_method='+'.join(methods))

    logger.info(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

    # Step 4: Compute metrics
    if G.number_of_nodes() > 0:
        # PageRank on reversed graph (high = foundational)
        try:
            pr = nx.pagerank(G.reverse(), alpha=0.85)
        except: pr = {n: 1/G.number_of_nodes() for n in G}
        bw = nx.betweenness_centrality(G)
        in_c = nx.in_degree_centrality(G)
        out_c = nx.out_degree_centrality(G)
        try: hubs, auths = nx.hits(G, max_iter=100)
        except: hubs = auths = {n: 0 for n in G}

        # Depth
        dag = G.copy()
        try:
            while True:
                cycle = nx.find_cycle(dag)
                weakest = min(cycle, key=lambda e: dag.edges[e[0], e[1]].get('confidence', 0))
                dag.remove_edge(weakest[0], weakest[1])
        except nx.NetworkXNoCycle: pass
        depth = {}
        for n in nx.topological_sort(dag):
            preds = list(dag.predecessors(n))
            depth[n] = max((depth.get(p, 0) for p in preds), default=-1) + 1 if preds else 0

        for n in G.nodes():
            G.nodes[n].update({
                'pagerank': round(pr.get(n, 0), 6),
                'betweenness': round(bw.get(n, 0), 4),
                'in_degree_centrality': round(in_c.get(n, 0), 4),
                'out_degree_centrality': round(out_c.get(n, 0), 4),
                'hub_score': round(hubs.get(n, 0), 4),
                'authority_score': round(auths.get(n, 0), 4),
                'depth': depth.get(n, 0),
            })

    # Step 5: Community detection (Louvain)
    if G.number_of_nodes() >= 2:
        try:
            communities = nx.community.louvain_communities(G.to_undirected(), seed=42)
        except:
            communities = [set(G.nodes())]
        for cid, members in enumerate(communities):
            for node in members:
                G.nodes[node]['community'] = cid
        G.graph['communities'] = {cid: sorted(m) for cid, m in enumerate(communities)}
        logger.info(f'Communities: {len(communities)} clusters found')
    else:
        for n in G: G.nodes[n]['community'] = 0
        G.graph['communities'] = {}

    return G


def get_learning_path(G):
    """Depth-first topological sort: foundations first."""
    g = G.copy()
    # Break cycles
    try:
        for cycle in list(nx.simple_cycles(g)):
            weakest = None
            wc = float('inf')
            for i in range(len(cycle)):
                u, v = cycle[i], cycle[(i+1) % len(cycle)]
                if g.has_edge(u, v):
                    c = g.edges[u, v].get('confidence', 0)
                    if c < wc: wc = c; weakest = (u, v)
            if weakest and g.has_edge(*weakest):
                g.remove_edge(*weakest)
    except: pass

    in_deg = {n: 0 for n in g}
    for _, v in g.edges(): in_deg[v] = in_deg.get(v, 0) + 1

    def sort_key(node):
        d = g.nodes[node]
        depth = d.get('depth', 0)
        out = g.out_degree(node)
        pr = d.get('pagerank', 0)
        ts = d.get('first_mention', '')
        if ts:
            parts = ts.split(':')
            try: secs = int(parts[0]) * 60 + int(parts[1])
            except: secs = 9999
        else: secs = 9999
        imp = d.get('importance_score', 0)
        return (depth, -out, -pr, secs, -imp, node)

    heap = []
    for n in g:
        if in_deg[n] == 0:
            heapq.heappush(heap, (sort_key(n), n))

    result = []
    while heap:
        _, node = heapq.heappop(heap)
        result.append(node)
        for _, nb in g.out_edges(node):
            in_deg[nb] -= 1
            if in_deg[nb] == 0:
                heapq.heappush(heap, (sort_key(nb), nb))

    remaining = [n for n in g if n not in set(result)]
    remaining.sort(key=sort_key)
    result.extend(remaining)
    return result


print('✅ Graph builder loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  VISUALIZATION — PyVis with community coloring
# ═══════════════════════════════════════════════════════════════

from pyvis.network import Network as PyVisNet

COMMUNITY_COLORS = ['#4dd0e1','#ff7043','#66bb6a','#ab47bc','#ffa726',
                    '#42a5f5','#ef5350','#26c6da','#8d6e63','#78909c']

def visualize_kg(G, height='600px'):
    """Render interactive knowledge graph HTML with community-colored nodes."""
    net = PyVisNet(height=height, width='100%', directed=True,
                   bgcolor='#0f1117', font_color='#e2e8f0')
    net.barnes_hut(gravity=-4000, central_gravity=0.3, spring_length=180,
                   spring_strength=0.04, damping=0.09)

    prs = [d.get('pagerank', 0) for _, d in G.nodes(data=True)]
    max_pr = max(prs) if prs and max(prs) > 0 else 1.0
    num_comm = len(G.graph.get('communities', {}))
    use_comm = num_comm > 1

    node_data, edge_data = {}, {}

    for node, d in G.nodes(data=True):
        pr = d.get('pagerank', 0)
        ratio = pr / max_pr if max_pr > 0 else 0.5
        size = 20 + ratio * 30
        comm = d.get('community', 0)
        color = COMMUNITY_COLORS[comm % len(COMMUNITY_COLORS)] if use_comm else f'#{int(77-44*ratio):02x}{int(208-58*ratio):02x}{int(225+18*ratio):02x}'
        name = d.get('normalized_name', node).replace('_', ' ').title()

        net.add_node(node, label=name, title='', size=size,
            color={'background': color, 'border': color,
                   'highlight': {'background': '#ffffff', 'border': '#ffffff'}},
            font={'size': 13, 'color': '#e2e8f0', 'strokeWidth': 3, 'strokeColor': '#0f1117'},
            borderWidth=2,
            shadow={'enabled': True, 'color': color, 'size': 14, 'x': 0, 'y': 0})

        node_data[node] = {
            'n': name, 'i': f"{d.get('importance_score',0):.2f}",
            'f': d.get('frequency', 0),
            't': d.get('timestamps', [])[:6],
            'p': round(ratio*100),
            'pr': f"{pr:.4f}", 'bw': f"{d.get('betweenness',0):.3f}",
            'dp': d.get('depth', 0),
            'cm': comm if use_comm else -1,
        }

    for i, (s, t, d) in enumerate(G.edges(data=True)):
        c = d.get('confidence', 0)
        eid = f'e{i}'
        op = 0.2 + c * 0.5
        net.add_edge(s, t, id=eid, title='', width=1+c*2.5,
            color={'color': f'rgba(148,163,184,{op:.2f})', 'highlight': '#90caf9'},
            arrows={'to': {'enabled': True, 'scaleFactor': 0.7}},
            smooth={'type': 'curvedCW', 'roundness': 0.15})
        sn = G.nodes[s].get('normalized_name', s).replace('_',' ').title()
        tn = G.nodes[t].get('normalized_name', t).replace('_',' ').title()
        edge_data[eid] = {'s': sn, 't': tn, 'c': f'{c:.2f}',
                          'm': d.get('detection_method',''), 'e': d.get('evidence','')[:100]}

    # Save and inject custom tooltips
    tmp = os.path.join(tempfile.gettempdir(), 'kg_viz.html')
    net.save_graph(tmp)
    with open(tmp, 'r') as f: html = f.read()

    html = re.sub(r'<center>\s*<h1>.*?</h1>\s*</center>\s*', '', html, count=1)
    html = html.replace('<body>', '<body style="margin:0;padding:0;overflow:hidden;background:#0f1117;">')

    tooltip_css = '''
<style>
div.vis-tooltip{display:none!important}
.kg-tt{display:none;position:fixed;z-index:99999;min-width:220px;max-width:340px;
background:rgba(15,17,26,0.97);backdrop-filter:blur(20px);border:1px solid rgba(99,179,237,0.18);
border-radius:16px;box-shadow:0 20px 60px rgba(0,0,0,0.55);color:#e2e8f0;
font-family:sans-serif;font-size:13px;pointer-events:none;animation:kgF 0.14s ease-out}
.kg-tt.ev{border-color:rgba(255,183,77,0.18)}
@keyframes kgF{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:translateY(0)}}
.kg-tt .th{padding:14px 18px 10px;border-bottom:1px solid rgba(255,255,255,0.06)}
.kg-tt .tn{font-size:16px;font-weight:600;color:#90caf9}
.kg-tt .tb{padding:10px 18px 14px}
.kg-tt .tr{display:flex;justify-content:space-between;padding:4px 0}
.kg-tt .tl{color:#64748b;font-size:11px;text-transform:uppercase;letter-spacing:0.6px}
.kg-tt .tv{color:#f1f5f9;font-weight:500}
.kg-tt .tp{width:100%;height:4px;background:rgba(255,255,255,0.06);border-radius:3px;margin-top:10px}
.kg-tt .tf{height:100%;border-radius:3px;background:linear-gradient(90deg,#4dd0e1,#2196f3,#5c6bc0)}
.kg-tt .eh{font-size:15px;font-weight:600;color:#ffb74d;padding:14px 18px 6px}
.kg-tt .ed{color:#94a3b8;font-size:12px;padding:2px 18px;line-height:1.5}
</style>'''

    tooltip_js = '''
<div class="kg-tt" id="ntt"></div>
<div class="kg-tt ev" id="ett"></div>
<script>
(function(){var ND=''' + json.dumps(node_data) + ''';var ED=''' + json.dumps(edge_data) + ''';
var ntt=document.getElementById('ntt'),ett=document.getElementById('ett');
function pos(el,ev){var x=ev.center?ev.center.x:0,y=ev.center?ev.center.y:0;
el.style.left=(x+18)+'px';el.style.top=(y+18)+'px';el.style.display='block';
var r=el.getBoundingClientRect();if(r.right>window.innerWidth-10)el.style.left=(x-r.width-18)+'px';
if(r.bottom>window.innerHeight-10)el.style.top=(y-r.height-18)+'px';}
function wait(){if(typeof network!=='undefined')init();else setTimeout(wait,80);}
function init(){network.setOptions({interaction:{tooltipDelay:999999,hover:true}});
network.on('hoverNode',function(p){var d=ND[p.node];if(!d)return;ett.style.display='none';
var h='<div class="th"><div class="tn">'+d.n+'</div></div><div class="tb">';
h+='<div class="tr"><span class="tl">Importance</span><span class="tv">'+d.i+'</span></div>';
h+='<div class="tr"><span class="tl">Mentions</span><span class="tv">'+d.f+'</span></div>';
h+='<div class="tr"><span class="tl">PageRank</span><span class="tv">'+d.pr+'</span></div>';
if(parseFloat(d.bw)>0)h+='<div class="tr"><span class="tl">Betweenness</span><span class="tv">'+d.bw+'</span></div>';
h+='<div class="tr"><span class="tl">Depth</span><span class="tv">'+d.dp+'</span></div>';
if(d.cm>=0)h+='<div class="tr"><span class="tl">Cluster</span><span class="tv">#'+(d.cm+1)+'</span></div>';
if(d.t.length)h+='<div class="tr"><span class="tl">Appears at</span><span class="tv">'+d.t.join(', ')+'</span></div>';
h+='<div class="tp"><div class="tf" style="width:'+d.p+'%"></div></div></div>';
ntt.innerHTML=h;pos(ntt,p.event);});
network.on('blurNode',function(){ntt.style.display='none';});
network.on('hoverEdge',function(p){var d=ED[p.edge];if(!d)return;ntt.style.display='none';
var h='<div class="eh">'+d.s+' \\u2192 '+d.t+'</div>';h+='<div class="ed">Confidence: '+d.c+'</div>';
if(d.m)h+='<div class="ed">Method: '+d.m+'</div>';
if(d.e)h+='<div class="ed" style="font-style:italic">\\u201c'+d.e+'\\u201d</div>';
ett.innerHTML=h;pos(ett,p.event);});
network.on('blurEdge',function(){ett.style.display='none';});
network.on('dragStart',function(){ntt.style.display='none';ett.style.display='none';});
network.on('zoom',function(){ntt.style.display='none';ett.style.display='none';});
}wait();})();</script>'''

    html = html.replace('</body>', tooltip_css + tooltip_js + '\n</body>')
    with open(tmp, 'w') as f: f.write(html)
    return tmp, html

print('✅ Visualization loaded')

## 4. Input — Choose Your Source

Run **one** of the cells below depending on your input type.

In [ ]:
# ─── Option A: YouTube URL ─────────────────────────────────
# Set YOUTUBE_URL above, then run this cell

transcript = None

if YOUTUBE_URL:
    print(f'📥 Downloading: {YOUTUBE_URL}')
    !yt-dlp -f 'bestaudio[ext=m4a]/bestaudio/best[height<=720]/best' --no-playlist -o '/tmp/yt_video.%(ext)s' "{YOUTUBE_URL}"
    import glob
    files = glob.glob('/tmp/yt_video.*')
    if files:
        video_path = files[0]
        print(f'🎤 Transcribing: {video_path}')
        transcript = transcribe_video(video_path)
        print(f'✅ Transcript: {len(transcript["segments"])} segments, {len(transcript["full_text"])} chars')
    else:
        print('❌ Download failed')
else:
    print('ℹ️ Set YOUTUBE_URL in the config cell above, then re-run this cell.')

In [ ]:
# ─── Option B: Upload Video File ──────────────────────────
from google.colab import files as colab_files

print('📎 Upload a video file (.mp4, .mkv, .webm, .avi, .mov):')
uploaded = colab_files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    print(f'🎤 Transcribing: {fname}')
    transcript = transcribe_video(fname)
    print(f'✅ Transcript: {len(transcript["segments"])} segments, {len(transcript["full_text"])} chars')
else:
    print('No file uploaded.')

In [ ]:
# ─── Option C: Paste Transcript Text ─────────────────────
# Set PASTE_TEXT in the config cell, or type directly below:

sample_text = PASTE_TEXT or """
Today we'll learn about sorting algorithms. First understand arrays - an array is a linear data structure
where elements are stored in contiguous memory. Then we'll learn about sorting.
Bubble sort is the simplest sorting algorithm - it repeatedly compares adjacent elements.
Merge sort uses divide and conquer approach. It divides the array into halves and merges them.
Recursion is used to implement merge sort. Time complexity of merge sort is O(n log n).
Quick sort also uses divide and conquer but with a pivot element.
Before learning merge sort, understand recursion and divide and conquer.
Binary search requires a sorted array. So sorting is prerequisite for binary search.
"""

# Telugu-English sample (uncomment to test):
# sample_text = """
# Ee roju manam data structures gurinchi nerchukundaam. Mundu arrays ardham chesuko tarvata linked lists chuddaam.
# Array ante oka linear data structure, ikkada elements contiguous memory lo store avuthayi.
# Linked list lo nodes pointers tho connect avuthayi. Stack ante LIFO data structure.
# Queue ante FIFO data structure. Stack kosam array use chesthaam.
# Mundu stack nerchukondi tarvata queue easy avuthundi.
# Tree oka hierarchical data structure. Binary tree lo each node ki maximum two children untayi.
# Graph ante nodes mariyu edges kaligi unna data structure.
# """

if sample_text.strip():
    transcript = text_to_transcript(sample_text.strip())
    print(f'✅ Transcript from text: {len(transcript["segments"])} segments')
else:
    print('No text provided.')

## 5. Run LLM Extraction

In [ ]:
assert transcript is not None, '❌ No transcript found. Run one of the input cells above first.'

print('🤖 Sending to Groq LLM for extraction...')
result = llm_extract(transcript)

concepts = result['concepts']
relationships = result['relationships']
domain = result['domain']

print(f'\n📚 Domain: {domain}')
print(f'🧠 Concepts extracted: {len(concepts)}')
for c in concepts:
    print(f"   • {c['name']:30s}  importance={c['importance_score']:.2f}  freq={c['frequency']}")

print(f'\n🔗 Relationships detected: {len(relationships)}')
for r in relationships:
    print(f"   {r['from']} → {r['to']}  (conf={r['confidence']:.2f})")

## 6. Build Knowledge Graph

In [ ]:
video_id = transcript.get('video_id', 'colab_video')
G = build_knowledge_graph(concepts, relationships, video_id)

# Display metrics
print(f'\n📊 Graph Metrics:')
print(f'   Nodes: {G.number_of_nodes()}')
print(f'   Edges: {G.number_of_edges()}')
print(f'   Density: {nx.density(G):.4f}')
communities = G.graph.get('communities', {})
print(f'   Communities: {len(communities)}')

print(f'\n🏆 Node Metrics:')
print(f"   {'Concept':30s} {'Depth':>5} {'PageRank':>10} {'Betweenness':>12} {'Community':>10}")
print(f"   {'─'*30} {'─'*5} {'─'*10} {'─'*12} {'─'*10}")
for n, d in sorted(G.nodes(data=True), key=lambda x: -x[1].get('pagerank', 0)):
    print(f"   {n:30s} {d.get('depth',0):5d} {d.get('pagerank',0):10.4f} {d.get('betweenness',0):12.4f} {d.get('community',0):10d}")

# Community detail
if len(communities) > 1:
    print(f'\n🧩 Topic Clusters:')
    for cid, members in communities.items():
        names = ', '.join(m.replace('_',' ').title() for m in sorted(members)[:8])
        print(f'   Cluster #{cid+1} ({len(members)} concepts): {names}')

## 7. Learning Path

In [ ]:
path = get_learning_path(G)

print('📚 Recommended Learning Order:')
print()
for i, node in enumerate(path, 1):
    d = G.nodes[node]
    name = d.get('normalized_name', node).replace('_', ' ').title()
    depth = d.get('depth', 0)
    comm = d.get('community', 0)
    print(f'  {i:2d}. {name:35s}  depth={depth}  cluster=#{comm+1}')

print()
print(' → '.join(n.replace('_',' ').title() for n in path))

## 8. Interactive Knowledge Graph Visualization

In [ ]:
from IPython.display import HTML, display

tmp_path, html = visualize_kg(G, height='650px')
display(HTML(html))
print(f'\n💾 Graph also saved to: {tmp_path}')

## 9. Export JSON Output

In [ ]:
# Build full JSON output
concepts_out = []
for i, (node, data) in enumerate(G.nodes(data=True), 1):
    concepts_out.append({
        'id': i, 'name': node,
        'normalized_name': data.get('normalized_name', node),
        'importance_score': data.get('importance_score', 0),
        'frequency': data.get('frequency', 0),
        'first_mention': data.get('first_mention', ''),
        'timestamps': data.get('timestamps', []),
        'pagerank': data.get('pagerank', 0),
        'betweenness': data.get('betweenness', 0),
        'depth': data.get('depth', 0),
        'community': data.get('community', 0),
    })
concepts_out.sort(key=lambda x: -x['importance_score'])

rels_out = []
for s, t, d in G.edges(data=True):
    rels_out.append({
        'from': s, 'to': t,
        'confidence': d.get('confidence', 0),
        'evidence': d.get('evidence', ''),
        'detection_method': d.get('detection_method', ''),
    })

output = {
    'video_id': video_id,
    'metadata': {'domain': domain, 'processed_at': now_iso(), 'extraction_method': 'llm'},
    'concepts': concepts_out,
    'relationships': rels_out,
    'learning_path': path,
    'analytics': {
        'total_concepts': len(concepts_out),
        'total_relationships': len(rels_out),
        'num_communities': len(G.graph.get('communities', {})),
        'graph_density': round(nx.density(G), 4),
    },
}

json_str = json.dumps(output, indent=2, ensure_ascii=False)
print(json_str[:3000])
if len(json_str) > 3000:
    print(f'\n... ({len(json_str)} total chars)')

# Save
out_path = f'/tmp/{video_id}_knowledge_graph.json'
with open(out_path, 'w') as f: f.write(json_str)
print(f'\n💾 Saved to: {out_path}')

In [ ]:
# ─── Download files ──────────────────────────────────────────
from google.colab import files as colab_files

# Download JSON
colab_files.download(out_path)

# Download graph HTML
colab_files.download(tmp_path)

---

## Summary of Enhancements

| Feature | Description |
|---|---|
| **Multi-Language Support** | English, Hindi-English (Hinglish), Telugu-English (Tenglish) — auto-detected |
| **Telugu Transliteration** | తెలుగు script converted to Roman for processing |
| **Overlapping Chunk Extraction** | Long transcripts split with 20% overlap for better boundary context |
| **Embedding Deduplication** | Sentence-transformer merges near-duplicate concepts (sim > 0.82) |
| **Edge Weight Refinement** | Multi-source edges merged with weighted average (LLM=0.5, pattern=0.3, etc.) |
| **Community Detection** | Louvain clustering identifies topic groups, shown as colored nodes |
| **Reversed PageRank** | PageRank on reversed graph highlights foundational (not complex) concepts |
| **Depth-first Learning Path** | Kahn's algorithm sorts by depth → out-degree → PageRank |